In [1]:
import torch
import os

from datetime import datetime

from itertools import product

from torch.utils.tensorboard import SummaryWriter


In [29]:
from dielectric_ml import train_predictors, data, models, utils

**LOAD DATA**

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df = data.process_exp_data("new_DE_data.xlsx", prop_col="Δε")
df = df.dropna(subset="PROP")

data_list, prop_scaler = data.predictor_dataloader(df, "PROP")

Processing molecules: 100%|██████████████████████████████████████████████████████| 3337/3337 [00:00<00:00, 5925.31it/s]


test: smiles and temps equal: (3337, 3337)


**TRAIN MODEL**

In [37]:
# set up config dictionary hyper parameters
n_conv_layers = [3]
dropouts = [0.15]
h_dims = [256]
learning_rates = [0.0005]
batch_sizes = [64]
epochs = [1]
n_splits = [2]

In [38]:
model_architectures = [models.Predictor]

In [39]:
prefix = "del_e"

In [40]:
for model_architecture in model_architectures:
    # Can now add optional prefix for classifier experiments
    configs = train_predictors.create_predictor_configs(
        model_architecture, 
        n_conv_layers, 
        dropouts, 
        h_dims, 
        learning_rates, 
        batch_sizes, 
        epochs, 
        n_splits, 
        prefix=prefix  # Optional: add prefix for this experiment
    )
    
    print(f"total configs to run {len(configs)}")

total configs to run 1


In [41]:
# create directory with prefix
os.makedirs(f"data/{prefix}_data", exist_ok=True)

In [42]:
# logging
results = {
      "batch_start_time": datetime.now().isoformat(),
     "batch_start_timestamp": datetime.now().timestamp()
    }

In [43]:
for i, config in enumerate(configs, 1):
            # run_classifier_experiment will use prefix from config
            writer_name, result = train_predictors.run_predictor_experiment(
                config=config, 
                model_type="Del_E_Predictor", 
                model_architecture=model_architecture, 
                data_list=data_list, 
                prop_scaler=prop_scaler,
                device=device
            )
            results[writer_name] = result
            utils.save_pickle(results=results, pickle_file=f"data/{prefix}_data/{model_architecture.__name__}_actuals_experiment.pkl")
            print(f"Results saved. Progress: {i}/{len(configs)}")

Predictor(
  (activation): ReLU()
  (gat_layers): ModuleList(
    (0): GATConv(79, 256, heads=8)
    (1-2): 2 x GATConv(2048, 256, heads=8)
  )
  (lin1): Linear(in_features=2048, out_features=256, bias=True)
  (lin2): Linear(in_features=256, out_features=1, bias=True)
)
running experiment: Predictor_n_3_d0.15_hdim_256_lr_0.0005_bs_64_e_1_s_2
starting fold 1 / 2...


  0%|                                                                                            | 0/1 [00:00<?, ?it/s]

fold: 1, epoch0/1train loss: 1.6612val loss: 1.0111rmse:18.4583


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:13<00:00, 13.23s/it]


completed fold 1/2
starting fold 2 / 2...


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:12<00:00, 12.20s/it]

fold: 2, epoch0/1train loss: 0.7872val loss: 1.0313rmse:18.0827
completed fold 2/2
Evaluating on hold out test set


final test loss: 1.0312
final test rmse: 17.8340
Training completed
saving model to models\del_e_Predictor\Predictor_n_3_d0.15_hdim_256_lr_0.0005_bs_64_e_1_s_2.pth
Results saved. Progress: 1/1
